In [1]:
import torch as pt


mols_test = pt.load('./data/mine/test_11499.pt')
print(len(mols_test))
mols_all = pt.load('./data/mine/mols_all.pt')
print(len(mols_all))

11499
2253216


In [2]:
# 统计词频
import numpy as np


mols_train = mols_all[:232826]


In [15]:
from torch.utils.data import DataLoader
from utils.data import SpecDataset
import numpy as np


def collate_fun_conv(batch):
    mzs_intens = []
    for mz, inten in batch:
        mz_inten = np.zeros(1000, dtype=np.float32)
        mz_inten[mz] = inten
        mzs_intens.append(mz_inten)
    return pt.tensor(np.array(mzs_intens), dtype=pt.float32)



dataset_lib = SpecDataset(mols_all)
dataset_test = SpecDataset(mols_test)
loader_lib = DataLoader(dataset_lib, batch_size=4096, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_conv)
loader_test = DataLoader(dataset_test, batch_size=4096, shuffle=False,
                        num_workers=8, collate_fn=collate_fun_conv)
dataset_train = SpecDataset(dataset_lib, mapping=np.arange(232826))
loader_train = DataLoader(dataset_train, batch_size=64, shuffle=True, 
                            num_workers=8, collate_fn=collate_fun_conv)
num_batches = len(loader_train)

In [18]:
import torch.optim as optim
import torch.nn as nn


class SpecConv(nn.Module):
    def __init__(self, spec_len:int=1000):
        super(SpecConv, self).__init__()
        self.linear1 = nn.Linear(spec_len, 500)
        self.linear2 = nn.Linear(500, 250)
        self.linear3 = nn.Linear(250, 125)
        self.encoder = nn.Sequential(self.linear1, nn.ReLU(), self.linear2, nn.ReLU(), self.linear3)
        self.linear4 = nn.Linear(125, 250)
        self.linear5 = nn.Linear(250, 500)
        self.linear6 = nn.Linear(500, spec_len)
        self.decoder = nn.Sequential(self.linear4, nn.ReLU(), self.linear5, nn.ReLU(), self.linear6)

    def forward(self, mzs_intens, mode:str='train'):
        if mode == 'train': 
            return self.decoder(self.encoder(mzs_intens))
        elif mode == 'emb': # emb模式下的masks只mask掉了padding 
            return self.encoder(mzs_intens)
        else:
            raise ValueError('mode not exist')


gpu = 6
model = SpecConv().to(gpu)

epochs = 10
lr = 0.001
optimizer = optim.Adam(model.parameters(), lr=lr)

In [19]:
from tqdm import tqdm
from utils.tools import build_idx, evaluate, save_model


def gen_embeddings(model:nn.Module, loader:DataLoader, gpu:int):
    model.eval()
    embs = []
    with pt.no_grad():
        for mzs_intens in loader:
            data = mzs_intens.to(gpu)
            emb = model(data, mode='emb').detach().cpu().numpy()
            embs.append(emb)
    pt.cuda.empty_cache()
    embs = np.concatenate(embs, axis=0)
    embs /= np.linalg.norm(embs, axis=1, keepdims=True)
    return embs


f = open('Linear.txt', 'w')
model_name = 'Linear'
max_metrics = {'expand': [0, 0], 'insilico': [0, 0]}
for epoch in range(epochs):
    print(f'==================================Train_epoch{epoch+1}======================================')
    model.train()
    train_loss = []
    for i, mzs_intens in enumerate(tqdm(loader_train, unit='batch')):
        data = mzs_intens.unsqueeze(1).to(gpu)
        optimizer.zero_grad()
        output = model(data)
        loss = ((output - data)**2).sum()
        train_loss.append(loss.item())
        loss.backward()
        optimizer.step()
        if (i+1) %300 ==0:
            loss = np.mean(train_loss)
            print(f'Total Loss: {loss}')
            train_loss = []
    
    print(f'===================================Test_epoch{epoch+1}======================================')
    f.write('\nTest_epoch%d\n' % (epoch+1))
    embeddings_lib = gen_embeddings(model, loader_lib, gpu)
    embeddings_test = gen_embeddings(model, loader_test, gpu)
    I_expand, _ = build_idx(embeddings_lib, embeddings_test, gpu)
    top1_expand, top10_expand = evaluate(mols_test, I_expand, mols_all, f, 'Expanded')
    if top1_expand > max_metrics['expand'][0] and top10_expand > max_metrics['expand'][1]:
        max_metrics['expand'] = [top1_expand, top10_expand]
        save_model(model, model_name, epoch)
    I_insilico, _ = build_idx(embeddings_lib[:2146690], embeddings_test, gpu)
    top1_insilico, top10_insilico = evaluate(mols_test, I_insilico, mols_all, f, 'In-silico')
    if top1_insilico > max_metrics['insilico'][0] and top10_insilico > max_metrics['insilico'][1]:
        max_metrics['insilico'] = [top1_insilico, top10_insilico]
        save_model(model, model_name, epoch)
    print(f'================================================================================================')
f.close()

==================================Train_epoch1======================================


 10%|▉         | 349/3638 [00:03<00:12, 272.64batch/s]

Total Loss: 186.07969472249349


 18%|█▊        | 673/3638 [00:04<00:07, 401.29batch/s]

Total Loss: 131.49872060139975


...

 99%|█████████▉| 3608/3638 [00:12<00:00, 357.10batch/s]

Total Loss: 27.30808115641276


100%|██████████| 3638/3638 [00:13<00:00, 279.10batch/s]

===================================Test_epoch10======================================


Searching time:  0:00:00.911072
Expanded library
Top1 hit rate: 11.26%
Top10 hit rate: 35.08%
Searching time:  0:00:00.858674
In-silico library
Top1 hit rate: 11.37%
Top10 hit rate: 35.38%
